## Notebook setup

In [ ]:
!pip install -q -U openai==1.56.1

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import openai
import json
import re
import time

import pandas as pd

# Performance API
#openai.api_key = "YOUR KEY"
#base_url = "https://genai.science-cloud.hu/performance-api/"

# Normal GenAI4Science API
openai.api_key = "sk-c6e3904c0dff418dbaeae7faa599af69"
base_url = "https://genai.science-cloud.hu/api/"

client = openai.OpenAI(base_url=base_url, api_key=openai.api_key)
print(f"Available models via GenAI4Science API:\n{[model['id'] for model in client.models.list().model_dump()['data']]}")

Available models via GenAI4Science API:
['codellama:13b', 'devstral-small-2:24b', 'codestral:22b', 'qwen3-coder:30b', 'deepseek-math7b', 'deepseek-r1:32b', 'gemma3:27b', 'gemma4:31b', 'gemma4-reasoning31b', 'gpt-oss:20b', 'gpt-oss:120b', 'llama3.1:8b', 'llama3.2:3b', 'llama3.3:70b', 'llama4:scout', 'llama4:maverick', 'mistral-small3.2:24b', 'qwen3.6:35b', 'qwen3.6:27b']


In [ ]:
# Select a model!
model = "gpt-oss:120b"

In [ ]:
def get_answer(user_prompt, model=model, max_retries=4):
    retry_delay = 60

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0,
            )

            if response is None:
                print("Response is None – retrying...")
                time.sleep(retry_delay)
                retry_delay *= 2
                continue

            # If choices is missing, skip retry and continue
            if not hasattr(response, "choices") or not response.choices:
                print("Missing choices in response – retrying...")
                time.sleep(retry_delay)
                retry_delay *= 2
                continue

            return response.choices[0].message.content

        except Exception as e:
            print(
                f"Request failed (attempt {attempt+1}/{max_retries}): {e}. "
                f"Retrying in {retry_delay} seconds..."
            )
            time.sleep(retry_delay)
            retry_delay *= 2

    # SKIP after max_retries
    print(f"Skipping prompt after {max_retries} failed attempts: {user_prompt[:120]}...")
    return None


## Load data

In [ ]:
df = pd.read_excel("ai3.xlsx")
df.head(1)

,cid,article,nem,születési_idő,halálozási_idő,vallás,születési_név,place_of_birth,place_of_death,prev_analysed,source_sheet
0,Q9312701,https://de.wikipedia.org/wiki/Roman_Zmorski,http://www.wikidata.org/entity/Q6581097,1822-08-09T00:00:00Z,1867-02-19T00:00:00Z,NaN,NaN,http://www.wikidata.org/entity/Q270,http://www.wikidata.org/entity/Q1731,NO,sr-author


In [ ]:
import requests
from bs4 import BeautifulSoup
import time

import time
import requests
from bs4 import BeautifulSoup
from requests.exceptions import RequestException

def get_wikipedia_text(url, max_retries=5, timeout=10):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/112.0.0.0 Safari/537.36'
    }
    retry_delay = 1

    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                content_div = soup.find('div', {'id': 'bodyContent'})
                if content_div:
                    paragraphs = content_div.find_all('p')
                    return '\n'.join(p.get_text(strip=True) for p in paragraphs)
                else:
                    return "Content area not found."
            elif response.status_code == 429:
                print(f"Rate limited (429). Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                retry_delay *= 2
            else:
                return f"Failed to fetch the page. Status code: {response.status_code}"
        except RequestException as e:
            print(f"Request failed: {e}. Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
            retry_delay *= 2

    return f"Failed to fetch the page after {max_retries} attempts."

## Scrape Wikipedia

In [ ]:
article_texts = []
lengths = []
urls = df.iloc[:, 1]

for url in urls:
    print(url)

    article_text = get_wikipedia_text(url)
    print(article_text[:1000])  # Print the first 1000 characters
    print('\n-----\n')

    article_texts.append(article_text)

    # --- Length classification ---
    char_count = len(article_text)
    if char_count < 300:
        length_label = "short"
    elif char_count <= 5000:
        length_label = "medium"
    else:
        length_label = "long"
    lengths.append(length_label)

    time.sleep(0.1)

https://de.wikipedia.org/wiki/Roman_Zmorski
Roman Zmorski(*9. August1822inWarschau,Kongresspolen; †19. Februar1867inDresden,Königreich Sachsen) war einpolnischerDichter, Zeitschriftenherausgeber und Volkskundler. Er lebte zeitweise in Russisch-Polen, Preußen, Frankreich, Sachsen,  im Osmanischen Reich und in Serbien.
Sein Vater war Richter. Roman Zmorski wuchs inWarschauundMasowienauf, wo er verschiedene Schulen besuchte. Vom Gymnasium in Warschau wurde er wegen konspirativer Tätigkeiten relegiert. Danach war er als Zeitschriftenredakteur tätig  und in verschiedenen literarischen Zirkeln in Warschau aktiv, darunter in der bekanntenCyganeria Warszawska(Warschauer Bohème).
Er war zwischenzeitlich auch in der Warschauer Zitadelle inhaftiert und lebte danach   an  wechselnden Orten.
1843 zog Roman Zmorski nachPreußen. Dort lebte er vor allem in der polnisch bewohntenProvinz Posen, hielt sich aber auch inSchlesienauf, wo er die dortige slawisch geprägte Kultur studierte. Er besuchte auch Ve

In [ ]:
df['article_text'] = article_texts
df['length'] = lengths

## Process Wikipedia articles

In [ ]:
prompt = """
TASK:
Analyze the given text and extract the following information in JSON format:

- Was the author a writer, poet, or playwright? Criteria for "writer" determination:
Before 1800: Any published text qualifies.
After 1800: Only fiction (novels, short stories, dramas, fairy tales) or poetry count as writing.
Memoirs, travel stories, screenplays, or song lyrics after 1800 do not count.

- What languages did the author publishing his/her works? (If the ARTICLE has no information about it then answer null!)
Try to assign only one language to an author!

Answer in JSON format only! Answer format:
{"writer": true/false, "language": null or list of ISO 639 language code}
Example answers:
{"writer": true, "language": ["en"]}
{"writer": false, "language": null}


ARTICLE:

{{article_text}}
"""

In [ ]:
answers = []
for url, text in zip(urls, article_texts):
  user_prompt = prompt.replace('{{article_text}}', text)
  answer = get_answer(user_prompt)
  answers.append(answer)

  print(url, '\n')
  print(answer)
  print('\n-----\n')

  time.sleep(10)

https://de.wikipedia.org/wiki/Roman_Zmorski 

{"writer": true, "language": ["pl"]}

-----

https://de.wikipedia.org/wiki/Laza_Kosti%C4%87 

{"writer": true, "language": ["sr"]}

-----

https://de.wikipedia.org/wiki/Momo_Kapor 

{"writer": true, "language": ["sr"]}

-----

https://de.wikipedia.org/wiki/Florian_Aaron 

{"writer": false, "language": ["ro"]}

-----

https://de.wikipedia.org/wiki/M%C3%B3zes_Kah%C3%A1na 

{"writer": true, "language": ["hu"]}

-----

https://de.wikipedia.org/wiki/Coloman_Braun-Bogdan 

{"writer": false, "language": ["ro"]}

-----

https://de.wikipedia.org/wiki/Constantin_Dobrogeanu-Gherea 

{"writer": false, "language": ["ro"]}

-----

https://de.wikipedia.org/wiki/Constantin_Virgil_Gheorghiu 

{"writer": true, "language": ["ro"]}

-----

https://de.wikipedia.org/wiki/Alexei_Mateevici 

{"writer": true, "language": ["ro"]}

-----

https://de.wikipedia.org/wiki/Corneliu_Vadim_Tudor 

{"writer": true, "language": ["ro"]}

-----

https://de.wikipedia.org/wiki/Da

In [ ]:
import json
import re
import pandas as pd

json_answers = []

# Regex pattern to capture the first JSON object in the text
pattern = re.compile(r'\{.*?\}', re.DOTALL)

for url, answer, length_label in zip(urls, answers, lengths):
    try:
        # Extract JSON substring
        json_text = pattern.search(answer).group(0)

        # Parse JSON
        json_answer = json.loads(json_text)

        # Add metadata
        json_answer['url'] = url
        json_answer['length'] = length_label

    except Exception as e:
        # Fallback entry if parsing fails
        json_answer = {
            'url': url,
            'length': length_label,
            'error': f'Failed to parse JSON: {e}'
        }

    json_answers.append(json_answer)

# Create dataframe
out_df = pd.DataFrame.from_records(json_answers)

# Show header (column names)
print(out_df.columns.tolist())

# Show first 25 rows
out_df.head(25)


['writer', 'language', 'url', 'length', 'error']


,writer,language,url,length,error
0,True,[pl],https://de.wikipedia.org/wiki/Roman_Zmorski,medium,NaN
1,True,[sr],https://de.wikipedia.org/wiki/Laza_Kosti%C4%87,medium,NaN
2,True,[sr],https://de.wikipedia.org/wiki/Momo_Kapor,medium,NaN
3,False,[ro],https://de.wikipedia.org/wiki/Florian_Aaron,medium,NaN
4,True,[hu],https://de.wikipedia.org/wiki/M%C3%B3zes_Kah%C...,medium,NaN
5,False,[ro],https://de.wikipedia.org/wiki/Coloman_Braun-Bo...,medium,NaN
6,False,[ro],https://de.wikipedia.org/wiki/Constantin_Dobro...,medium,NaN
7,True,[ro],https://de.wikipedia.org/wiki/Constantin_Virgi...,medium,NaN
8,True,[ro],https://de.wikipedia.org/wiki/Alexei_Mateevici,medium,NaN
9,True,[ro],https://de.wikipedia.org/wiki/Corneliu_Vadim_T...,medium,NaN


In [ ]:
out_df.to_excel('de-ai3.xlsx', index=False)

In [ ]:
from google.colab import files
files.download('de-ai3.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>